# Production-Grade RAG System
## Multi-PDF · LangChain · LangGraph · ChromaDB · RAGAS | Fully Local with Ollama

---

## The Scenario

Say we work at a small research lab where people keep asking questions across three
foundational AI papers:

| Paper | Why it's hard to parse |
|-------|--------------|
| *Attention Is All You Need* | Architecture diagrams, attention score tables, BLEU benchmarks |
| *BERT* | Pre-training task tables, NLP benchmark comparisons, figures |
| *RAG Paper* | Retrieval architecture, QA tables, knowledge base charts |

The goal: build a RAG system that finds answers across all three papers — text, tables,
and images — and can measure its own answer quality.

---

## What We Build (Full Architecture)

```
3 Complex PDFs (Text + Tables + Images)
        │
        ├─ Text      → PyMuPDF extraction → text chunks
        ├─ Tables    → pdfplumber → markdown tables
        │              ↳ FALLBACK: Camelot → Docling → Vision LLM
        └─ Images    → metadata stored
                       ↳ UPGRADE: Vision LLM description → CLIP embedding
                |
        RecursiveCharacterTextSplitter
                |
        qwen3-embedding:4b → ChromaDB (persistent)
                |
     ┌──────────┼────────────────────────────┐
     │          │                            │
  Basic RAG  Advanced RAG           Agentic RAG (LangGraph)
  (chain)    Multi-Query            CRAG: route→retrieve
             HyDE                   →grade→generate
             BM25+Semantic          →check→self-correct
             Compression
                |
        RAGAS Evaluation
        Faithfulness · Relevancy · Precision · Recall
```

---
# Chapter 1 — Setup & Configuration

## Why Ollama?

```
OpenAI / Google API:          Ollama (our choice):
  ✅ Easy                       ✅ 100% free, runs offline
  ✅ Powerful models            ✅ Data never leaves your machine
  ❌ Costs money per call       ✅ No rate limits
  ❌ Data sent to cloud         ✅ Works in air-gapped environments
  ❌ Rate limits                ❌ Needs good hardware (8GB+ RAM)
```

## Models We'll Use

| Model | Role | Why |
|-------|------|-----|
| `qwen3.5:9b` | Answer generation, routing, grading | Strong reasoning, tool-calling support |
| `qwen3-embedding:4b` | Convert text → vectors | High-quality 2048-dim embeddings |

> **Before running**: Make sure Ollama is running (`ollama serve`) and
> you've pulled both models (`ollama pull qwen3.5:9b`, `ollama pull qwen3-embedding:4b`).

### Cell below: Install all required Python packages

| Package | What it does |
|---------|-------------|
| `langchain-ollama` | Connect LangChain to Ollama LLMs and embeddings |
| `langchain-chroma` | LangChain wrapper for ChromaDB vector store |
| `langchain-community` | BM25Retriever, EnsembleRetriever for hybrid search |
| `rank_bm25` | The BM25 algorithm implementation |
| `pymupdf` | Fast PDF text + image extraction (import as `fitz`) |
| `pdfplumber` | Accurate PDF table extraction |
| `ragas` | RAG evaluation framework |
| `datasets` | Required by RAGAS internally |

In [3]:
# ── Install all required packages ──────────────────────────────────────────────
import subprocess, sys

packages = [
    "langchain-ollama",        # Ollama LLM + embeddings integration
    "langchain-chroma",        # ChromaDB vectorstore integration
    "langchain-community",     # BM25Retriever, EnsembleRetriever
    "rank_bm25",               # BM25 algorithm for hybrid search
    "pymupdf",                 # PDF text + image extraction (fitz)
    "pdfplumber",              # PDF table extraction
    "pillow",                  # Image processing
    "ragas",                   # RAG evaluation framework
    "datasets",                # HuggingFace datasets (used by RAGAS)
    "langchain-text-splitters",# Text chunking utilities
    "matplotlib",              # Results visualization
    "pandas",                  # Data display
]

print("Installing packages...")
for pkg in packages:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    status = "✅" if r.returncode == 0 else "❌"
    print(f"  {status} {pkg}")

print("\n✅ All packages ready!")

### Cell below: Configuration — change these to match your setup

**Key variables explained:**

- `LLM_MODEL`: The Ollama model for generating answers. Change to `qwen3.5:4b` if 9b isn't downloaded yet.
- `CHROMA_PATH`: Where ChromaDB saves vectors to disk. Persistent = no re-embedding on restart.
- `CHUNK_SIZE`: How many characters per text chunk. 800 is a good default for research papers.
- `CHUNK_OVERLAP`: How many chars to repeat between consecutive chunks. Prevents losing context at boundaries.

```
CHUNK_SIZE=800, OVERLAP=150 means:

  chunk 1: chars 0–800
  chunk 2: chars 650–1450   ← 150 chars overlap with chunk 1
  chunk 3: chars 1300–2100  ← 150 chars overlap with chunk 2

  Why overlap? If a key sentence lands at position 795,
  it appears in BOTH chunk 1 and chunk 2 → never lost.
```

In [4]:
# ── Production Configuration ────────────────────────────────────────────────────
import os

# ─── Ollama Models ────────────────────────────────────────────────────────────
# Change LLM_MODEL to match what you pulled: ollama pull qwen3.5:9b
LLM_MODEL   = "qwen3.5:4b"           # Primary LLM (change to qwen3.5:4b if 9b not ready)
EMBED_MODEL = "qwen3-embedding:4b"   # Embedding model (ollama pull qwen3-embedding:4b)

# ─── Research Papers (publicly available, complex PDFs) ───────────────────────
os.makedirs("./research_pdfs", exist_ok=True)

RESEARCH_PAPERS = [
    {
        "id"    : "attention",
        "title" : "Attention Is All You Need",
        "url"   : "https://arxiv.org/pdf/1706.03762",
        "path"  : "./research_pdfs/attention.pdf",
        "topic" : "Transformer Architecture",
    },
    {
        "id"    : "bert",
        "title" : "BERT: Pre-training of Deep Bidirectional Transformers",
        "url"   : "https://arxiv.org/pdf/1810.04805",
        "path"  : "./research_pdfs/bert.pdf",
        "topic" : "Language Model Pre-training",
    },
    {
        "id"    : "rag_paper",
        "title" : "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "url"   : "https://arxiv.org/pdf/2005.11401",
        "path"  : "./research_pdfs/rag_paper.pdf",
        "topic" : "Retrieval-Augmented Generation",
    },
]

# ─── ChromaDB ─────────────────────────────────────────────────────────────────
CHROMA_PATH     = "./chroma_research_db"   
COLLECTION_NAME = "research_papers"

# ─── Chunking ─────────────────────────────────────────────────────────────────
CHUNK_SIZE    = 800    
CHUNK_OVERLAP = 150   

# ─── Retrieval ────────────────────────────────────────────────────────────────
TOP_K        = 5
MAX_RETRIES  = 2   

print("Configuration:")
print(f"  LLM Model       : {LLM_MODEL}")
print(f"  Embedding Model : {EMBED_MODEL}")
print(f"  ChromaDB Path   : {CHROMA_PATH}")
print(f"  Chunk Size      : {CHUNK_SIZE} chars / {CHUNK_OVERLAP} overlap")
print(f"  Top-K Retrieval : {TOP_K}")
print(f"\n  Papers to index : {len(RESEARCH_PAPERS)}")
for p in RESEARCH_PAPERS:
    print(f"    [{p['id']}] {p['title'][:55]}...")

### Cell below: Health check — verify Ollama is running and models are ready

If you see ❌ here, the most common fixes:
```bash
# Ollama not running:
ollama serve

# Model not downloaded:
ollama pull qwen3.5:9b
ollama pull qwen3-embedding:4b

# Wrong model name — list what you have:
ollama list
```

In [5]:
# ── Ollama Health Check ─────────────────────────────────────────────────────────
# Verify both models are available before we invest time in PDF processing
import ollama

print("Checking Ollama models...\n")

# List available models
try:
    available = [m.model for m in ollama.list().models]
    print("Available models:")
    for m in available:
        print(f"  • {m}")
    print()
except Exception as e:
    print(f"❌ Cannot connect to Ollama: {e}")
    print("   Make sure Ollama is running: ollama serve")

# Test LLM
print(f"Testing {LLM_MODEL}...")
try:
    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "Say: LLM ready"}],
        options={"num_predict": 10}
    )
    print(f"  ✅ {LLM_MODEL} → {resp.message.content.strip()[:50]}")
except Exception as e:
    print(f"  ❌ {LLM_MODEL}: {e}")
    print(f"     Run: ollama pull {LLM_MODEL}")

# Test embeddings
print(f"\nTesting {EMBED_MODEL}...")
try:
    emb = ollama.embeddings(model=EMBED_MODEL, prompt="test")
    print(f"  ✅ {EMBED_MODEL} → {len(emb.embedding)}-dim vector")
except Exception as e:
    print(f"  ❌ {EMBED_MODEL}: {e}")
    print(f"     Run: ollama pull {EMBED_MODEL}")

print("\n✅ Health check complete.")

---
# Chapter 2 — PDF Processing Pipeline

## The Core Problem

A PDF is not a document — it is a **canvas with positioned objects**.
There is no concept of 'paragraph', 'table', or 'figure' inside a PDF file.
Everything is just text characters at (x, y) coordinates, or pixel blobs for images.

```
What a PDF ACTUALLY stores:

  'Revenue' at x=50,  y=100
  '$4.2M'   at x=200, y=100
  'Q3'      at x=300, y=100
  'APAC'    at x=50,  y=120
  '$5.1M'   at x=200, y=120
  'Q4'      at x=300, y=120
  [image blob: 480×320px at x=50, y=200]

Our job: reconstruct meaning from these raw coordinates.
```

## Three Content Types — Three Different Problems

| Content | Challenge | Our Tool |
|---------|-----------|----------|
| **Text** | Easy — just concatenate chars in reading order | PyMuPDF `page.get_text()` |
| **Tables** | Hard — columns collapse into flat string | pdfplumber (grid detection) |
| **Images** | Harder — pixels have no text representation | metadata now; Vision LLM as upgrade |

## Why We Process All Three Separately

If we only use `page.get_text()` (the naive approach):
```
Table in PDF:              What get_text() returns:
  | Model | EN-DE |          'Model EN-DE Transformer 28.4 LSTM 26.1'
  | Trans | 28.4  |          → all on one line, no column structure
  | LSTM  | 26.1  |          → embedding model cannot understand relationships

Image in PDF:              What get_text() returns:
  [Architecture diagram]    '' (empty string — images have no text layer)
```

By processing separately, every piece of content gets into ChromaDB
in the format best understood by the embedding model.

### Cell below: Download the 3 research PDFs

We use arxiv.org papers because they are:
- Freely available (open access)
- Complex (text + tables + figures)
- Highly relevant (the papers are about the very models we're using)

> **Expected time**: 30-60 seconds per paper depending on internet speed.
> Papers are saved to `./research_pdfs/` and skipped on subsequent runs.

In [6]:
# ── Download PDFs ───────────────────────────────────────────────────────────────
import requests, os, time

def download_pdf(url: str, save_path: str, title: str) -> bool:
    if os.path.exists(save_path):
        size_kb = os.path.getsize(save_path) // 1024
        print(f"  ✅ Already exists: {os.path.basename(save_path)} ({size_kb} KB)")
        return True

    print(f"  ⬇️  Downloading: {title}")
    headers = {"User-Agent": "Mozilla/5.0 (academic research)"}
    try:
        r = requests.get(url, headers=headers, timeout=90, stream=True)
        r.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        size_kb = os.path.getsize(save_path) // 1024
        print(f"     ✅ Saved ({size_kb} KB)")
        return True
    except Exception as e:
        print(f"     ❌ Failed: {e}")
        return False


print("Downloading research papers...\n")
success_count = 0
for paper in RESEARCH_PAPERS:
    ok = download_pdf(paper["url"], paper["path"], paper["title"])
    if ok:
        success_count += 1
    time.sleep(1)   # polite delay between requests

print(f"\n→ {success_count}/{len(RESEARCH_PAPERS)} papers ready")

### Cell below: Inspect each PDF before extracting

Always inspect before extracting. This tells you:
- How many pages to expect
- Which pages are text-heavy vs image-heavy
- Whether there are annotations (highlights, comments) to handle

> **What the numbers mean:**
> - Text chars: total characters extracted. < 100 per page = likely scanned or image-only.
> - Images: number of embedded images. Includes small icons — we'll filter by size later.

In [7]:
# ── Inspect PDF Structure ───────────────────────────────────────────────────────
# Before extracting, understand what we're dealing with
import fitz  

print("PDF Structure Analysis")
print("=" * 65)

for paper in RESEARCH_PAPERS:
    if not os.path.exists(paper["path"]):
        print(f"  [SKIP] {paper['title']} — not downloaded")
        continue

    doc = fitz.open(paper["path"])
    total_pages = len(doc)

    # Count content types across all pages
    total_text_chars = 0
    total_images     = 0

    for page in doc:
        total_text_chars += len(page.get_text())
        total_images     += len(page.get_images(full=True))

    doc.close()

    print(f"\n  📄 {paper['title'][:50]}")
    print(f"     Pages      : {total_pages}")
    print(f"     Text chars : {total_text_chars:,}")
    print(f"     Images     : {total_images}")
    print(f"     Topic      : {paper['topic']}")

print("\n" + "=" * 65)

---
## Step 1 — Text Extraction

### How PyMuPDF Works

```python
doc  = fitz.open('paper.pdf')
page = doc[0]
text = page.get_text()   # reads internal text layer in reading order
```

PyMuPDF reads the PDF's internal text layer (the character position data).
It reconstructs reading order using the y-coordinate (top to bottom) then x-coordinate (left to right).

### What Good vs Bad Output Looks Like

```
GOOD (digital PDF):
  'The dominant sequence transduction models are based on complex
   recurrent or convolutional neural networks...'

BAD (scanned PDF — image of a page):
  ''   ← empty string! The text layer doesn't exist.
       ← If this happens, you need OCR (EasyOCR, Tesseract)
```

### Metadata Strategy

Every chunk carries metadata so we can filter later:
```python
metadata = {
    'paper_id'     : 'attention',     # for filtering by paper
    'page'         : 3,               # for citation
    'content_type' : 'text',          # for filtering by type
}
# Usage later:
# vectorstore.similarity_search(q, filter={'paper_id': 'bert'})
# vectorstore.similarity_search(q, filter={'content_type': 'table'})
```

In [8]:
# ── Step 1: Text Extraction (PyMuPDF) ──────────────────────────────────────────
#
# WHY PyMuPDF over pdfplumber for text?
#   → Faster, handles complex layouts better, preserves reading order
#   → pdfplumber excels at TABLES but text quality is similar
#
# METADATA STRATEGY:
#   Every chunk gets: source, paper_id, page_num, content_type
#   This lets us filter ChromaDB by paper or content type later

import fitz

def extract_text_chunks(paper: dict) -> list:
    """Extract text from each page of a PDF with rich metadata."""
    if not os.path.exists(paper["path"]):
        return []

    doc    = fitz.open(paper["path"])
    chunks = []

    for page_num, page in enumerate(doc):
        text = page.get_text().strip()

        chunks.append({
            "content"  : text,
            "metadata" : {
                "source"       : paper["path"],
                "paper_id"     : paper["id"],
                "paper_title"  : paper["title"],
                "topic"        : paper["topic"],
                "page"         : page_num + 1,
                "content_type" : "text",
                "char_count"   : len(text),
            }
        })

    doc.close()
    return chunks


all_text_chunks = []
print("Extracting text from PDFs...\n")

for paper in RESEARCH_PAPERS:
    chunks = extract_text_chunks(paper)
    all_text_chunks.extend(chunks)
    total_chars = sum(c["metadata"]["char_count"] for c in chunks)
    print(f"  [{paper['id']:10s}] {len(chunks):2d} pages | {total_chars:,} chars total")

print(f"\n→ Total text chunks: {len(all_text_chunks)}")
print(f"\nSample (attention paper, page 1):")
sample = next(c for c in all_text_chunks if c["metadata"]["paper_id"] == "attention")
print(sample["content"][:400])

---
## Step 2 — Table Extraction

### Why Tables Break Naive Text Extraction

```
Table in PDF:               page.get_text() gives:
  | Model     | EN-DE |      'Model EN-DE EN-FR Transformer 28.4 41.0 LSTM 26.1 39.2'
  | Transformer | 28.4 |     ← flat string, no column relationships
  | LSTM      | 26.1  |     ← embedding model cannot answer 'BLEU score of Transformer on EN-DE'

pdfplumber gives:           We convert to markdown:
  [['Model','EN-DE'],         | Model       | EN-DE |
   ['Transformer','28.4'],    |-------------|-------|
   ['LSTM','26.1']]           | Transformer | 28.4  |
                              | LSTM        | 26.1  |
```

### Why Markdown Format for Tables?

The embedding model (`qwen3-embedding`) was trained on text from the internet,
which includes huge amounts of GitHub README files, documentation, and Stack Overflow answers
— all of which use markdown tables.

The `|` pipe characters act as **visual separators** the model understands:
it knows `28.4` belongs to `Transformer` and column `EN-DE`.

CSV would also work but is noisier:
```
CSV: 'Model,EN-DE,Transformer,28.4'  ← commas mix header with data
MD:  '| Transformer | 28.4 |'        ← pipes clearly group values
```

### How pdfplumber Detects Tables

pdfplumber looks for **grid lines** (horizontal and vertical rules) drawn in the PDF.
It finds intersections → reconstructs cell boundaries → fills with text.

> **IMPORTANT**: This only works when the table has **visible grid lines**.
> Many academic paper tables use NO borders (borderless style).
> We handle this in the very next section.

In [9]:
# ── Step 2: Table Extraction (pdfplumber) ───────────────────────────────────────
#
# WHY tables need special treatment:
#   PyMuPDF text extraction: "BLEU EN-DE EN-FR ... 28.4 41.0 38.1 ..."  (garbled!)
#   pdfplumber table extract: proper list-of-lists → markdown table (clean!)
#
# EMBEDDING TABLES AS MARKDOWN:
#   | Model     | EN-DE | EN-FR |
#   |-----------|-------|-------|
#   | Transformer | 28.4 | 41.0 |
#
#   The | separators help the embedding model understand column relationships

import pdfplumber

def table_to_markdown(table: list) -> str:
    """Convert pdfplumber table (list of lists) to markdown string."""
    if not table or len(table) < 2:
        return ""

    rows_md = []
    for row in table:
        cells = [str(c).strip().replace("\n", " ") if c else "" for c in row]
        rows_md.append("| " + " | ".join(cells) + " |")

    # Insert separator after header
    sep = "| " + " | ".join(["---"] * len(table[0])) + " |"
    rows_md.insert(1, sep)
    return "\n".join(rows_md)


def extract_table_chunks(paper: dict) -> list:
    """Extract all tables from a PDF as markdown chunks."""
    if not os.path.exists(paper["path"]):
        return []

    chunks = []
    with pdfplumber.open(paper["path"]) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for tbl_idx, table in enumerate(tables):
                md = table_to_markdown(table)
                if not md or len(md) < 50:
                    continue

                chunks.append({
                    "content"  : md,
                    "metadata" : {
                        "source"       : paper["path"],
                        "paper_id"     : paper["id"],
                        "paper_title"  : paper["title"],
                        "topic"        : paper["topic"],
                        "page"         : page_num + 1,
                        "table_index"  : tbl_idx,
                        "content_type" : "table",
                        "rows"         : len(table),
                    }
                })
    return chunks


all_table_chunks = []
print("Extracting tables from PDFs...\n")

for paper in RESEARCH_PAPERS:
    chunks = extract_table_chunks(paper)
    all_table_chunks.extend(chunks)
    print(f"  [{paper['id']:10s}] {len(chunks)} tables found")

print(f"\n→ Total table chunks: {len(all_table_chunks)}")

if all_table_chunks:
    sample = all_table_chunks[0]
    print(f"\nSample table (page {sample['metadata']['page']}):")
    print(sample["content"][:500])

---
## ⚠️ What To Do When Table Extraction Fails

pdfplumber will return an empty list `[]` in these 5 situations:

```
SCENARIO 1 — Borderless table (most common in research papers)
  Model       EN-DE   EN-FR
  Transformer  28.4    41.0     ← no grid lines drawn
  LSTM         26.1    39.2     ← pdfplumber returns []

SCENARIO 2 — Merged/complex headers
  ┌──────────┬─────────────────┐
  │          │    Results      │  ← one cell spans two columns
  │  Model   ├────────┬────────┤  ← pdfplumber misreads structure
  │          │ EN-DE  │ EN-FR  │
  └──────────┴────────┴────────┘

SCENARIO 3 — Scanned PDF (image of a document)
  The page IS a JPEG/PNG. No text layer. pdfplumber returns ''.
  Need OCR first, then table detection.

SCENARIO 4 — Table split across two pages
  Page 4: Header row only
  Page 5: Data rows only
  → Two incomplete tables. Must be stitched.

SCENARIO 5 — Two-column PDF layout
  Table sits inside left column.
  pdfplumber reads across both columns → garbled output.
```

## Solutions in Order of Complexity

```
Try 1 → pdfplumber   (fast, handles bordered tables)
Try 2 → Camelot      (handles borderless via whitespace alignment)
Try 3 → Docling      (ML model — handles merged cells, complex layouts)
Try 4 → Vision LLM   (last resort — reads the page as an image)
```

In [10]:
# ══════════════════════════════════════════════════════════════════════════
# SOLUTION 1: Camelot — Best for Borderless Tables
# ══════════════════════════════════════════════════════════════════════════
#
# Camelot has TWO flavours:
#   'lattice' — uses grid lines, same as pdfplumber (no advantage)
#   'stream'  — uses whitespace gaps to detect column boundaries
#                → THIS is the one that handles borderless tables
#
# Install: pip install camelot-py[cv]   (needs opencv-python too)
#
# HOW STREAM MODE WORKS:
#   'Model       EN-DE   EN-FR'   ← 7 spaces between 'Model' and 'EN-DE'
#   Camelot sees: gap > threshold → column boundary → splits here
#   Result: ['Model', 'EN-DE', 'EN-FR']
#
# ACCURACY SCORE:
#   Camelot scores itself (0-100%) based on text alignment.
#   > 90%  → trust the result
#   70-90% → review manually
#   < 70%  → use fallback (Docling or Vision LLM)

# Uncomment to install:
# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'camelot-py[cv]', '-q'])

def extract_tables_camelot(pdf_path: str, page_num: int) -> list[str]:
    """
    Extract tables from a single PDF page using Camelot stream mode.
    Returns list of markdown strings. Returns [] if nothing found or quality is low.
    """
    try:
        import camelot

        # Stream mode: detect columns by whitespace gaps
        tables = camelot.read_pdf(
            pdf_path,
            pages  = str(page_num + 1),  # camelot uses 1-based page numbers
            flavor = 'stream',           # whitespace-based column detection
            edge_tol = 50,               # tolerance for aligning text to columns
        )

        results = []
        for t in tables:
            print(f"  [Camelot] Page {page_num+1}: accuracy={t.accuracy:.1f}%")

            if t.accuracy < 70:
                print(f"    ⚠️  Low confidence — skipping (use Docling or Vision LLM)")
                continue

            # Convert DataFrame to markdown
            md = t.df.to_markdown(index=False)
            results.append(md)
            print(f"    ✅ Extracted {t.df.shape[0]} rows × {t.df.shape[1]} cols")
            print(f"    Preview: {md[:200]}")

        return results

    except ImportError:
        print("  [Camelot] not installed. Run: pip install camelot-py[cv]")
        return []
    except Exception as e:
        print(f"  [Camelot] failed: {e}")
        return []


print("extract_tables_camelot() defined.")
print()
print("When to use Camelot stream:")
print("  - pdfplumber returned []")
print("  - PDF has NO visible border lines on tables")
print("  - Text is cleanly spaced in columns (most academic papers)")


In [11]:
# ══════════════════════════════════════════════════════════════════════════
# SOLUTION 2: Docling — Best for Complex/Merged Tables
# ══════════════════════════════════════════════════════════════════════════
#
# Docling (IBM Research, 2024) uses an ML model trained on document layouts.
# It understands: merged cells, nested headers, spanning rows, multi-line cells.
#
# HOW IT WORKS:
#   1. Runs a layout detection model (like YOLO but for documents)
#   2. Classifies regions: Title | Paragraph | Table | Figure | List
#   3. For Table regions: runs a table structure model
#   4. Exports to clean markdown preserving the logical structure
#
# COMPARISON:
#   pdfplumber: rule-based (grid lines)  → fast, fragile
#   Camelot   : rule-based (whitespace)  → medium, fragile for complex
#   Docling   : ML-based                 → slower, robust for complex layouts
#
# WHEN TO USE:
#   - Merged header cells (e.g., 'Performance' spanning 3 columns)
#   - Multi-line cell content
#   - Tables inside tables
#   - When Camelot produces garbage

# Uncomment to install:
# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'docling', '-q'])

def extract_tables_docling(pdf_path: str) -> list[str]:
    """
    Extract ALL tables from an entire PDF using Docling.
    Returns list of markdown strings (one per table found).
    Note: processes whole file, not page-by-page.
    """
    try:
        from docling.document_converter import DocumentConverter

        print(f"  [Docling] Converting {pdf_path}...")
        converter = DocumentConverter()
        result    = converter.convert(pdf_path)

        tables = []
        for i, table in enumerate(result.document.tables):
            md = table.export_to_markdown()
            if md and len(md) > 30:
                tables.append(md)
                print(f"  [Docling] Table {i+1}: {len(md)} chars")

        print(f"  [Docling] Found {len(tables)} tables total")
        return tables

    except ImportError:
        print("  [Docling] not installed. Run: pip install docling")
        return []
    except Exception as e:
        print(f"  [Docling] failed: {e}")
        return []


print("extract_tables_docling() defined.")
print()
print("Docling vs pdfplumber comparison:")
print("  Feature              pdfplumber   Camelot   Docling")
print("  Bordered tables      ✅           ✅         ✅")
print("  Borderless tables    ❌           ✅         ✅")
print("  Merged cells         ❌           ❌         ✅")
print("  Multi-line cells     ❌           ❌         ✅")
print("  Speed                Fast         Medium     Slow")
print("  GPU required         No           No         No (CPU ok)")


In [12]:
# ══════════════════════════════════════════════════════════════════════════
# SOLUTION 3: Vision LLM — Last Resort, Always Works
# ══════════════════════════════════════════════════════════════════════════
#
# When all parsers fail, convert the page to an IMAGE and send it
# to a multimodal LLM (Vision LLM). The LLM 'reads' the image like
# a human would and describes the table in markdown.
#
# SCENARIOS WHERE THIS IS THE ONLY OPTION:
#   - Scanned PDFs (no text layer at all)
#   - Tables as screenshots inside the PDF
#   - Complex colour-coded tables
#   - Tables inside figures
#
# TRADE-OFFS:
#   ✅ Works on ANYTHING the model can see
#   ✅ Can interpret colours, arrows, annotations
#   ❌ SLOW: 10-60 seconds per page
#   ❌ Can hallucinate numbers (especially in dense tables)
#   ❌ Needs a Vision-capable model (qwen2.5vl, llava, etc.)

import fitz, io, base64
from PIL import Image

def extract_table_with_vision_llm(
    pdf_path : str,
    page_num : int,
    model    : str = 'qwen2.5vl:3b',  # change to llava:7b if qwen-vl unavailable
    dpi_scale: float = 2.0            # higher = sharper image, more VRAM needed
) -> str:
    """
    Convert ONE PDF page to a high-res image, send to Vision LLM,
    ask it to extract tables as markdown.

    Returns: markdown string, or 'No tables found.' if none detected.
    """
    import ollama

    # ── Step 1: Render page to PNG ─────────────────────────────────────────
    doc  = fitz.open(pdf_path)
    page = doc[page_num]

    # Matrix scales the rendering: 2.0 = 2× zoom = ~144 DPI
    # Higher DPI = more detail = better for dense tables
    mat = fitz.Matrix(dpi_scale, dpi_scale)
    pix = page.get_pixmap(matrix=mat)
    img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
    doc.close()

    # ── Step 2: Base64 encode for Ollama API ──────────────────────────────
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()

    # ── Step 3: Prompt the Vision LLM ─────────────────────────────────────
    # Specific prompt gets much better results than vague ones
    prompt = """
    Examine this PDF page carefully.
    Find ALL tables on this page.
    For EACH table:
      1. Extract every row and column exactly as shown
      2. Format it as a markdown table with | separators
      3. Preserve headers, units, and footnotes

    If there are NO tables on this page, respond with exactly: No tables found.

    Output ONLY the markdown tables. No explanation text.
    """

    try:
        resp = ollama.chat(
            model    = model,
            messages = [{'role': 'user', 'content': prompt, 'images': [b64]}]
        )
        return resp.message.content.strip()
    except Exception as e:
        return f'Vision LLM error: {e}'


print("extract_table_with_vision_llm() defined.")
print()
print("Vision LLM table extraction — use when:")
print("  - PDF is scanned (no text layer)")
print("  - Camelot and Docling both return garbage")
print("  - Table has colour coding or complex visual formatting")
print()
print("Supported vision models via Ollama:")
print("  ollama pull qwen2.5vl:3b   # 4GB VRAM, good quality")
print("  ollama pull llava:7b       # 6GB VRAM, good quality")
print("  ollama pull moondream      # 2GB VRAM, basic quality")


In [13]:
# ══════════════════════════════════════════════════════════════════════════
# PRODUCTION PATTERN: Smart Fallback Pipeline
# ══════════════════════════════════════════════════════════════════════════
#
# In production, you never rely on one method.
# Try fast methods first, fall back to slow-but-robust only when needed.
#
# Decision flow:
#
#   pdfplumber found tables?  ──YES──► use them ✅
#          │NO
#          ▼
#   Camelot accuracy > 70%?   ──YES──► use them ✅
#          │NO or error
#          ▼
#   Tables critical for user? ──YES──► Vision LLM (slow but always works)
#          │NO
#          ▼
#   Skip tables on this page (text chunks still available)

def extract_tables_with_fallback(
    pdf_path : str,
    page_num : int,
    use_vision_fallback: bool = False,  # set True to enable Vision LLM
    vision_model: str = 'qwen2.5vl:3b'
) -> list[str]:
    """
    Try pdfplumber → Camelot → Vision LLM in order.
    Returns list of markdown table strings.
    """

    # ── Method 1: pdfplumber (fast, bordered tables) ───────────────────────
    import pdfplumber
    with pdfplumber.open(pdf_path) as pdf:
        page   = pdf.pages[page_num]
        tables = page.extract_tables()

    if tables:
        results = []
        for t in tables:
            if not t or len(t) < 2:
                continue
            rows_md = []
            for row in t:
                cells = [str(c).strip().replace('\n', ' ') if c else '' for c in row]
                rows_md.append('| ' + ' | '.join(cells) + ' |')
            sep = '| ' + ' | '.join(['---'] * len(t[0])) + ' |'
            rows_md.insert(1, sep)
            md = '\n'.join(rows_md)
            if len(md) > 50:
                results.append(md)
        if results:
            print(f"  Page {page_num+1}: pdfplumber found {len(results)} table(s) ✅")
            return results

    # ── Method 2: Camelot stream (borderless tables) ───────────────────────
    results = extract_tables_camelot(pdf_path, page_num)
    if results:
        return results

    # ── Method 3: Vision LLM (last resort) ────────────────────────────────
    if use_vision_fallback:
        print(f"  Page {page_num+1}: using Vision LLM fallback (slow)...")
        vision_result = extract_table_with_vision_llm(
            pdf_path, page_num, model=vision_model
        )
        if 'No tables found' not in vision_result and '|' in vision_result:
            print(f"  Page {page_num+1}: Vision LLM extracted table ✅")
            return [vision_result]

    print(f"  Page {page_num+1}: no tables found (or extraction failed)")
    return []


print("Smart fallback pipeline defined.")
print()
print("Usage:")
print("  # Basic (no vision fallback — fast):")
print("  tables = extract_tables_with_fallback('paper.pdf', page_num=5)")
print()
print("  # With Vision LLM fallback (slower but handles anything):")
print("  tables = extract_tables_with_fallback(")
print("      'paper.pdf', page_num=5,")
print("      use_vision_fallback=True,")
print("      vision_model='qwen2.5vl:3b'")
print("  )")


---
## Step 3 — Handling Images in PDFs

### The Image Problem in RAG

Research papers are full of figures that carry critical information:

```
Figure 1 (Attention Is All You Need):
  [Transformer architecture diagram showing encoder-decoder with
   multi-head attention, feed-forward layers, positional encoding]

  Someone asks: 'How does the encoder connect to the decoder?'
  The answer is IN the diagram — but pure text extraction returns ''.
```

### Four Strategies (choose based on your needs)

```
STRATEGY 1 — Skip images entirely
  Pros: Zero extra work, fast
  Cons: All image content is lost
  Use when: Images are decorative (logos, headers) or irrelevant

STRATEGY 2 — Store metadata (page + size)
  Pros: Fast, enables page-level citations
  Cons: Cannot answer questions about image content
  Use when: Images exist but you can't afford Vision LLM latency
  ← THIS is what our current notebook uses

STRATEGY 3 — Vision LLM description
  Pros: Full image content indexed, answers figure questions
  Cons: Slow (10-60s per image), may hallucinate
  Use when: Figures contain key information (architecture diagrams,
            result charts, process flows)

STRATEGY 4 — CLIP visual embedding
  Pros: Images embedded directly (no text bottleneck)
  Cons: Needs PyTorch, query must also use CLIP, harder to debug
  Use when: Large image collections, images are the primary content
```

In [14]:
# ── Step 3: Image Metadata Extraction (PyMuPDF) ─────────────────────────────────
#
# APPROACH: We store image location metadata (page, bounding box, size)
# WITHOUT using a Vision LLM to describe content.
#
# WHY: Adding Vision LLM adds latency + The image chunks give us figure-level citations.
#
# TO ADD Vision LLM descriptions: see multimodal_rag_approaches.ipynb
#
# WHAT WE STORE:
#   content  = "[Figure on page X: image W×H px — topic]"
#   metadata = page, paper_id, bounding box, size

import fitz

def extract_image_metadata(paper: dict, min_px: int = 80) -> list:
    """Extract image location metadata from a PDF."""
    if not os.path.exists(paper["path"]):
        return []

    doc    = fitz.open(paper["path"])
    chunks = []

    for page_num, page in enumerate(doc):
        for img_idx, img_info in enumerate(page.get_images(full=True)):
            xref = img_info[0]
            try:
                base_img = doc.extract_image(xref)
                w = base_img["width"]
                h = base_img["height"]

                if w < min_px or h < min_px:
                    continue   # skip icons/decorations

                # Create a text description the embedding model can index
                content = (
                    f"[Figure {img_idx + 1} on page {page_num + 1} of '{paper['title']}'. "
                    f"Image size: {w}×{h} pixels. "
                    f"Topic: {paper['topic']}. "
                    f"This figure appears in the section covering {paper['topic']}. "
                    f"For detailed figure analysis, refer to page {page_num + 1}.]"
                )

                chunks.append({
                    "content"  : content,
                    "metadata" : {
                        "source"       : paper["path"],
                        "paper_id"     : paper["id"],
                        "paper_title"  : paper["title"],
                        "topic"        : paper["topic"],
                        "page"         : page_num + 1,
                        "img_index"    : img_idx,
                        "content_type" : "image",
                        "width"        : w,
                        "height"       : h,
                    }
                })
            except Exception:
                continue

    doc.close()
    return chunks


all_image_chunks = []
print("Extracting image metadata from PDFs...\n")

for paper in RESEARCH_PAPERS:
    chunks = extract_image_metadata(paper)
    all_image_chunks.extend(chunks)
    print(f"  [{paper['id']:10s}] {len(chunks)} images (≥80px)")

print(f"\n→ Total image metadata chunks: {len(all_image_chunks)}")

# Combined summary
print("\n── Content Summary ──────────────────────────────")
print(f"  Text pages  : {len(all_text_chunks)}")
print(f"  Tables      : {len(all_table_chunks)}")
print(f"  Images      : {len(all_image_chunks)}")
print(f"  TOTAL raw   : {len(all_text_chunks) + len(all_table_chunks) + len(all_image_chunks)}")

In [15]:
# ══════════════════════════════════════════════════════════════════════════
# STRATEGY 3: Vision LLM — Describe Images for Full Content Indexing
# ══════════════════════════════════════════════════════════════════════════
#
# HOW IT WORKS:
#   PDF page → extract image bytes → base64 encode
#   → send to Vision LLM with structured prompt
#   → LLM returns text description
#   → embed that description as a normal text chunk
#
# RESULT:
#   Query: 'How does the encoder connect to the decoder?'
#   Hits chunk: 'Figure 1: Encoder-decoder architecture. Encoder output
#   is passed to cross-attention layers in each decoder block...'
#   ← this would NEVER be retrieved without Vision LLM
#
# STRUCTURED PROMPT IS KEY:
#   Bad prompt:  'Describe this image'
#                → vague, inconsistent, misses key data
#   Good prompt: 'Is this a diagram/chart/table/photo?'
#                'List all labels, values, axes, relationships'
#                'What concept does this illustrate?'
#                → consistent, detailed, retrievable

import fitz, io, base64
from PIL import Image

VISION_DESCRIBE_PROMPT = """
You are analyzing an image extracted from an AI/ML research paper.

Answer these questions about the image:
1. TYPE: Is it a (diagram / chart / graph / table / photo / equation / other)?
2. CONTENT: What does it show? List all labels, axes, values, and relationships.
3. CONCEPT: What ML/AI concept does it illustrate?
4. KEY NUMBERS: List any important numbers, percentages, or metrics shown.

Be specific and technical. This description will be used for search retrieval,
so include all detail that a researcher might search for.
"""

def describe_image_with_vision_llm(
    pil_image   : Image.Image,
    model       : str = 'qwen2.5vl:3b',
    page_num    : int = 0,
    paper_title : str = ''
) -> str:
    """
    Send a PIL image to Vision LLM and get a detailed text description.
    Returns the description string to be stored as a chunk.
    """
    import ollama

    # Encode image to base64
    buf = io.BytesIO()
    pil_image.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()

    try:
        resp = ollama.chat(
            model    = model,
            messages = [{
                'role'    : 'user',
                'content' : VISION_DESCRIBE_PROMPT,
                'images'  : [b64]
            }]
        )
        description = resp.message.content.strip()

        # Prefix with location info for better retrieval
        return (
            f"[Figure on page {page_num} of '{paper_title}'.]\n"
            f"{description}"
        )
    except Exception as e:
        w, h = pil_image.size
        return f"[Figure on page {page_num}: {w}x{h}px. Vision LLM unavailable: {e}]"


def extract_images_with_descriptions(
    paper     : dict,
    model     : str = 'qwen2.5vl:3b',
    min_px    : int = 80,
    max_images: int = 5   # limit per paper to control latency
) -> list[dict]:
    """
    Extract images from PDF and describe each using Vision LLM.
    Returns list of chunk dicts (same format as text/table chunks).

    max_images: safety limit — Vision LLM is slow, cap at N per paper.
    """
    import fitz
    import os

    if not os.path.exists(paper['path']):
        return []

    doc    = fitz.open(paper['path'])
    chunks = []
    count  = 0

    for page_num, page in enumerate(doc):
        if count >= max_images:
            break

        for img_info in page.get_images(full=True):
            if count >= max_images:
                break

            xref = img_info[0]
            try:
                base_img = doc.extract_image(xref)
                w, h     = base_img['width'], base_img['height']

                if w < min_px or h < min_px:
                    continue  # skip tiny icons/logos

                pil_img  = Image.open(io.BytesIO(base_img['image'])).convert('RGB')
                print(f"  Describing image {count+1} on page {page_num+1} ({w}x{h})...")

                description = describe_image_with_vision_llm(
                    pil_img,
                    model       = model,
                    page_num    = page_num + 1,
                    paper_title = paper['title']
                )

                chunks.append({
                    'content' : description,
                    'metadata': {
                        'source'       : paper['path'],
                        'paper_id'     : paper['id'],
                        'paper_title'  : paper['title'],
                        'topic'        : paper['topic'],
                        'page'         : page_num + 1,
                        'content_type' : 'image_described',  # different from 'image'
                        'width'        : w,
                        'height'       : h,
                        'extractor'    : 'vision_llm',
                    }
                })
                count += 1

            except Exception as e:
                print(f"    Skipped: {e}")

    doc.close()
    return chunks


print("Vision LLM image description functions defined.")
print()
print("To use (adds Vision LLM descriptions to your pipeline):")
print("  # Pull a vision model first:")
print("  # ollama pull qwen2.5vl:3b")
print()
print("  image_chunks_described = []")
print("  for paper in RESEARCH_PAPERS:")
print("      chunks = extract_images_with_descriptions(paper, model='qwen2.5vl:3b')")
print("      image_chunks_described.extend(chunks)")
print()
print("  # Then add to all_documents before embedding:")
print("  all_documents += [Document(page_content=c['content'], metadata=c['metadata'])")
print("                    for c in image_chunks_described]")


In [16]:
# ══════════════════════════════════════════════════════════════════════════
# STRATEGY 4: CLIP — Embed Images Directly (Reference / Advanced)
# ══════════════════════════════════════════════════════════════════════════
#
# CLIP (Contrastive Language-Image Pre-training, OpenAI 2021) learns a
# SHARED vector space for text AND images.
#
# This means:
#   embed('a diagram of transformer architecture')  →  [0.23, -0.14, ...]  (512-dim)
#   embed(transformer_diagram_image.png)            →  [0.21, -0.13, ...]  (512-dim)
#   cosine_similarity(text_vec, image_vec) = 0.94  ← very similar!
#
# This is the foundation of ColPali (2024) — the current state-of-the-art
# for visual document retrieval.
#
# COMPARISON TO STRATEGY 3 (Vision LLM):
#
#   Strategy 3 (Vision LLM → text → embed):
#     Image → VLM describes → text → qwen3-embedding → 2048-dim
#     Pro: Uses same ChromaDB, works with existing pipeline
#     Con: Description is a LOSSY summary (detail gets dropped)
#
#   Strategy 4 (CLIP → image embed):
#     Image → CLIP → 512-dim  (stored separately)
#     Text query → CLIP → 512-dim → compare
#     Pro: No information loss — the full image is embedded
#     Con: Needs separate vector store or ChromaDB collection
#          Text and image embeddings use DIFFERENT DIMENSIONS
#          (qwen3-embed = 2048-dim, CLIP = 512-dim → cannot mix!)

# ── Reference implementation (needs: pip install transformers torch) ──────────

def embed_images_with_clip(pil_images: list) -> 'numpy.ndarray':
    """
    Generate 512-dim CLIP embeddings for a list of PIL images.
    These live in the same vector space as CLIP text embeddings.

    Requires: pip install transformers torch torchvision
    """
    from transformers import CLIPProcessor, CLIPModel
    import torch

    model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
    processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

    inputs = processor(images=pil_images, return_tensors='pt', padding=True)
    with torch.no_grad():
        feats = model.get_image_features(**inputs)   # shape: (N, 512)

    return feats.numpy()


def embed_texts_with_clip(texts: list) -> 'numpy.ndarray':
    """
    Generate 512-dim CLIP embeddings for text.
    MUST use this (not qwen3-embedding) to search over CLIP image embeddings.
    Text and image are in the SAME vector space — direct comparison works.
    """
    from transformers import CLIPProcessor, CLIPModel
    import torch

    model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
    processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

    inputs = processor(text=texts, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        feats = model.get_text_features(**inputs)   # also 512-dim

    return feats.numpy()


print("CLIP strategy reference code defined (needs transformers + torch to run).")
print()
print("Strategy comparison summary:")
print()
print("  Strategy | Implementation | Speed  | Quality | Complexity")
print("  ---------|----------------|--------|---------|----------")
print("  1 Skip   | Nothing        | Fast   | N/A     | None")
print("  2 Meta   | PyMuPDF only   | Fast   | Low     | Low  ← current notebook")
print("  3 VLM    | Vision LLM     | Slow   | High    | Medium")
print("  4 CLIP   | CLIP model     | Medium | High    | High")
print()
print("Recommendation for this course: Strategy 3 (Vision LLM)")
print("Add it when you have: qwen2.5vl:3b pulled + need figure-level answers")


---
# Chapter 3 — Intelligent Chunking Strategies

## Why We Can't Embed Whole Pages

```
Page 1 of Attention paper: ~3,000 characters

If we embed the whole page:
  embed('Introduction... The dominant sequence transduction models...
         ... multi-head attention ... encoder-decoder ...')
  → one 2048-dim vector that 'averages out' all content

  Query: 'How many attention heads?'
  → vector similarity to the WHOLE page is moderate
  → query might match a page that just mentions attention
     without answering the specific 'how many' question

If we embed a 800-char chunk:
  embed('The model uses h=8 parallel attention heads. For each head,
         we use dk = dv = dmodel/h = 64 dimensions...')
  → vector is SPECIFIC to attention heads
  → this chunk wins the similarity search decisively
```

## The Goldilocks Problem

```
TOO SMALL (< 200 chars):              TOO LARGE (> 2000 chars):
  'attention heads'                     Whole page about 3 topics
  → no context, just a phrase           → embedding 'averages out'
  → retrieval: precise but useless      → retrieval: noisy, low precision
  → LLM cannot answer from fragment     → LLM loses focus in prompt

JUST RIGHT (600-1000 chars):
  Complete paragraph or two about one concept
  → enough context to understand, specific enough to retrieve
```

## Why Recursive Splitter > Fixed Splitter

```python
RecursiveCharacterTextSplitter tries separators in order:
  1. '\n\n'  ← split on paragraph break (best)
  2. '\n'   ← split on line break
  3. '. '   ← split on sentence end
  4. ' '    ← split on word boundary
  5. ''     ← split on character (last resort)

Result: always splits at the most natural boundary available.
Fixed splitter always cuts at exactly 800 chars → mid-sentence.
```

In [17]:
# ── Chunking Strategy Comparison ────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

sample_text = all_text_chunks[0]["content"] if all_text_chunks else "Sample text not available."

print("Chunking Strategy Comparison")
print("=" * 60)
print(f"Input text : {len(sample_text):,} characters\n")

strategies = [
    {
        "name"    : "Recursive Character Split (RECOMMENDED)",
        "splitter": RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
    },
]

for s in strategies:
    chunks = s["splitter"].split_text(sample_text)
    sizes  = [len(c) for c in chunks]
    print(f"  {s['name']}")
    print(f"    Chunks       : {len(chunks)}")
    print(f"    Avg size     : {sum(sizes)//len(sizes)} chars")
    print(f"    Size range   : {min(sizes)}–{max(sizes)} chars")
    print(f"    Sample chunk : {chunks[1][:120].strip()}...")
    print()

In [18]:
# ── Apply Chunking to All Documents ─────────────────────────────────────────────
#
# RULE: Only split TEXT chunks (pages). Keep tables and image metadata whole.
#  - Tables: splitting mid-table destroys row/column relationships
#  - Images: already short descriptive strings

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

all_documents: list[Document] = []

# ── Split text chunks ──────────────────────────────────────────────────────────
text_split_count = 0
for chunk in all_text_chunks:
    if len(chunk["content"]) > CHUNK_SIZE:
        sub_texts = splitter.split_text(chunk["content"])
        for i, sub in enumerate(sub_texts):
            meta = {**chunk["metadata"], "sub_chunk": i, "total_sub_chunks": len(sub_texts)}
            all_documents.append(Document(page_content=sub, metadata=meta))
        text_split_count += len(sub_texts)
    else:
        all_documents.append(Document(
            page_content=chunk["content"], metadata=chunk["metadata"]
        ))
        text_split_count += 1

# ── Keep table and image chunks whole ─────────────────────────────────────────
for chunk in all_table_chunks + all_image_chunks:
    all_documents.append(Document(
        page_content=chunk["content"], metadata=chunk["metadata"]
    ))

# ── Summary ───────────────────────────────────────────────────────────────────
from collections import Counter
type_counts  = Counter(d.metadata["content_type"] for d in all_documents)
paper_counts = Counter(d.metadata["paper_id"]     for d in all_documents)

print("Final Document Chunks")
print("=" * 50)
print(f"  text   : {type_counts.get('text', 0):>4} chunks")
print(f"  table  : {type_counts.get('table', 0):>4} chunks")
print(f"  image  : {type_counts.get('image', 0):>4} chunks")
print(f"  TOTAL  : {len(all_documents):>4} chunks")
print()
print("By paper:")
for paper_id, count in paper_counts.items():
    print(f"  {paper_id:12s}: {count} chunks")

---
# Chapter 4 — Embedding & ChromaDB Vector Store

## What Is an Embedding?

An embedding is a list of numbers (a vector) that represents the **meaning** of text.
Texts with similar meaning have vectors that point in similar directions.

```
'The transformer uses multi-head attention'  → [0.23, -0.14, 0.88, 0.05, ...]  (2048 numbers)
'The model employs h parallel attention fns' → [0.22, -0.13, 0.86, 0.06, ...]  ← very close!
'The weather in Mumbai is hot and humid'     → [-0.12, 0.45, -0.33, 0.71, ...]  ← far away

cosine_similarity(sent1, sent2) = 0.97  ← nearly identical direction
cosine_similarity(sent1, sent3) = 0.12  ← very different direction
```

This is how RAG retrieval works: embed the query, find the closest stored vectors.

## Why ChromaDB?

```
ChromaDB vs Alternatives:

  ChromaDB:   ✅ Runs locally (no server needed)
               ✅ Persistent to disk (no re-embedding)
               ✅ Metadata filtering built-in
               ✅ LangChain integration out of the box
               ❌ Not designed for billions of vectors (use Pinecone/Weaviate)

  FAISS:      ✅ Very fast
               ❌ No metadata filtering
               ❌ Manual save/load

  Pinecone:   ✅ Cloud-native, scales to billions
               ❌ Paid, sends data to cloud

  ChromaDB is perfect for: local dev, teaching, < 1M documents.
```

## Persistent Storage — The Key Production Habit

```python
# BAD (ephemeral — re-embeds every run):
vectorstore = Chroma.from_documents(docs, embedding=emb)

# GOOD (persistent — loads from disk on restart):
vectorstore = Chroma(
    persist_directory='./chroma_db',
    embedding_function=emb
)
vectorstore.add_documents(docs)   # saved to disk automatically
# Next run: just open it, no re-embedding needed
```

Embedding 200 chunks takes ~2-5 minutes. With persistence,
you only pay that cost ONCE. Every subsequent session loads in seconds.

In [19]:
# ── Embedding Setup ──────────────────────────────────────────────────────────────
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=EMBED_MODEL)

# Test embedding dimensions and quality
print(f"Embedding model: {EMBED_MODEL}")
print()

test_texts = [
    "The transformer model uses multi-head self-attention.",
    "BERT uses a bidirectional encoder representation.",
    "RAG combines retrieval with generation.",
]

import numpy as np

test_embeddings = embeddings.embed_documents(test_texts)
dim = len(test_embeddings[0])
print(f"  Embedding dimensions: {dim}")

# Compute cosine similarities (higher = more similar)
def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("\n  Semantic similarity test:")
print(f"  transformer ↔ BERT      : {cosine_sim(test_embeddings[0], test_embeddings[1]):.3f}  (should be moderate)")
print(f"  transformer ↔ RAG       : {cosine_sim(test_embeddings[0], test_embeddings[2]):.3f}  (should be lower)")
print(f"  BERT ↔ RAG              : {cosine_sim(test_embeddings[1], test_embeddings[2]):.3f}")
print()
print("✅ Embeddings working correctly")

In [20]:
# ── ChromaDB — Persistent Vector Store ─────────────────────────────────────────
#
# PRODUCTION TIP: Use persist_directory to save vectors to disk.
# On restart, just load — no re-embedding. Saves 5-10 min for large corpora.
#
# INCREMENTAL INDEXING: Check if collection exists before re-creating.

from langchain_chroma import Chroma
import shutil

# Check if DB already exists with correct number of documents
rebuild = True
if os.path.exists(CHROMA_PATH):
    try:
        existing_db = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=CHROMA_PATH
        )
        existing_count = existing_db._collection.count()
        print(f"Found existing ChromaDB with {existing_count} vectors.")
        if existing_count >= len(all_documents) * 0.9:   # within 10% of expected
            vectorstore = existing_db
            rebuild = False
            print("  ✅ Using existing database (no re-embedding needed)")
        else:
            print(f"  ⚠️  Count mismatch (expected ~{len(all_documents)}). Rebuilding...")
            shutil.rmtree(CHROMA_PATH)
    except Exception as e:
        print(f"  ⚠️  Cannot load existing DB: {e}. Rebuilding...")
        shutil.rmtree(CHROMA_PATH, ignore_errors=True)

if rebuild:
    print(f"Building ChromaDB from {len(all_documents)} chunks...")
    print("  (This calls the embedding model once per chunk — may take 2-5 min)")
    print()

    # Batch into groups of 50 to show progress
    batch_size = 50
    batches = [all_documents[i:i+batch_size] for i in range(0, len(all_documents), batch_size)]

    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=CHROMA_PATH
    )

    for i, batch in enumerate(batches):
        vectorstore.add_documents(batch)
        print(f"  Batch {i+1}/{len(batches)} done ({min((i+1)*batch_size, len(all_documents))}/{len(all_documents)} chunks)")

total = vectorstore._collection.count()
print(f"\n✅ ChromaDB ready — {total} vectors stored at {CHROMA_PATH}")

In [21]:
# ── Verify Retrieval + Metadata Filtering ───────────────────────────────────────
print("Retrieval Verification\n")

test_query = "How does multi-head attention work?"

# ── A) Standard similarity search ─────────────────────────────────────────────
print(f"Query: '{test_query}'")
results = vectorstore.similarity_search_with_score(test_query, k=3)
print("\nTop 3 results (all papers):")
for doc, score in results:
    paper  = doc.metadata.get("paper_id", "?")
    page   = doc.metadata.get("page", "?")
    ctype  = doc.metadata.get("content_type", "?")
    print(f"  [{paper} | p{page} | {ctype}] score={score:.4f}")
    print(f"  {doc.page_content[:120].strip()}...")
    print()

# ── B) Filter by paper ────────────────────────────────────────────────────────
print("─" * 55)
print("Filtered to 'attention' paper only:")
filtered = vectorstore.similarity_search(
    test_query, k=2,
    filter={"paper_id": "attention"}
)
for doc in filtered:
    print(f"  [p{doc.metadata['page']} | {doc.metadata['content_type']}] {doc.page_content[:100]}...")

# ── C) Filter by content type ─────────────────────────────────────────────────
print("\nFiltered to tables only:")
table_results = vectorstore.similarity_search(
    "model performance BLEU score results", k=3,
    filter={"content_type": "table"}
)
for doc in table_results:
    print(f"  [{doc.metadata['paper_id']} | p{doc.metadata['page']}]")
    print(f"  {doc.page_content[:200]}")
    print()

---
# Chapter 5 — Basic RAG Chain

## The RAG Formula

```
RAG = Retrieval + Augmented + Generation

Retrieval:   similarity_search(query) → top-5 relevant chunks
Augmented:   inject those chunks into the LLM prompt as context
Generation:  LLM reads context + generates answer grounded in it

The LLM never generates from memory alone.
It READS the retrieved passages and answers from them.
```

## Why RAG Over Fine-tuning?

```
Fine-tuning:            RAG:
  Bakes knowledge        Retrieves knowledge at query time
  into model weights     ↳ update docs without retraining
  Expensive to update    Cheap to add new documents
  Fixed at train time    Always uses latest information
  No source citation     Full source traceability
```

## The Prompt Engineering Trick

We label each retrieved chunk with its type and source:
```
[TEXT from 'Attention Is All You Need', Page 3]
We propose a new simple network architecture, the Transformer...

[TABLE from 'Attention Is All You Need', Page 8]
| Model | EN-DE | EN-FR |
|-------|-------|-------|
| Trans | 28.4  | 41.0  |

This lets the LLM:
  1. Know WHERE the information comes from
  2. Include citations in its answer
  3. Give more weight to tables for numerical questions
```

> **Baseline to beat**: Once basic RAG is working, we measure it with RAGAS
> (Chapter 8) to get baseline scores, then improve with Advanced RAG (Chapter 6).

In [22]:
# ── Basic RAG Chain ─────────────────────────────────────────────────────────────
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── LLM Setup ─────────────────────────────────────────────────────────────────
llm = ChatOllama(model=LLM_MODEL, temperature=0, num_ctx=8192)

# ── Retriever ─────────────────────────────────────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

# ── RAG Prompt ────────────────────────────────────────────────────────────────

RAG_PROMPT = ChatPromptTemplate.from_template("""
You are a research assistant for TechMind Research Institute.
Answer the question using ONLY the provided context.
Always cite which paper and page number your answer comes from.
If the context doesn't contain enough information, say so clearly.

Context (from research papers):
{context}

Question: {question}

Answer (include citations like [paper_title, page X]):
""")


def format_docs(docs):
    """Format retrieved docs with metadata labels for the LLM."""
    parts = []
    for doc in docs:
        paper = doc.metadata.get("paper_title", "Unknown Paper")
        page  = doc.metadata.get("page", "?")
        ctype = doc.metadata.get("content_type", "text").upper()
        parts.append(f"[{ctype} from '{paper}', Page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


# ── Build Chain ───────────────────────────────────────────────────────────────
basic_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("✅ Basic RAG chain built.")
print(f"   Retriever : top-{TOP_K} chunks by cosine similarity")
print(f"   LLM       : {LLM_MODEL}")
print(f"   Context   : formatted with paper + page citations")

In [23]:
# ── Test Basic RAG ──────────────────────────────────────────────────────────────
TEST_QUESTIONS = [
    "How many attention heads does the original Transformer model use, and why?",
    "What is the difference between BERT-Base and BERT-Large?",
    "How does RAG handle questions about facts not seen during training?",
]

print("Basic RAG — Test Results")
print("=" * 70)

for q in TEST_QUESTIONS:
    print(f"\nQ: {q}")
    print("-" * 70)

    # Retrieve and show source documents
    docs = retriever.invoke(q)
    print("Retrieved chunks:")
    for doc in docs:
        paper = doc.metadata.get("paper_id", "?")
        page  = doc.metadata.get("page", "?")
        ctype = doc.metadata.get("content_type", "?")
        icon  = {"text": "📄", "table": "📊", "image": "🖼️"}.get(ctype, "📄")
        print(f"  {icon} [{paper} p{page}] {doc.page_content[:80].strip()}...")

    answer = basic_rag_chain.invoke(q)
    print(f"\nAnswer: {answer}")
    print("=" * 70)

---
# Chapter 6 — Advanced RAG Techniques

## The Core Problem with Basic RAG

Basic RAG has one critical weakness:

```
If the user's phrasing doesn't match the chunk's phrasing → retrieval fails.

User asks:   'How many parallel attention functions are there?'
Chunk says:  'The model uses h=8 heads'

Cosine similarity between these is LOW because:
  'parallel attention functions' ≠ 'h=8 heads'  (different vocabulary)
→ this chunk is NOT retrieved → LLM cannot answer
```

## Four Techniques, Four Different Problems

```
MULTI-QUERY:
  Problem: one phrasing misses relevant chunks
  Fix: generate 3 query variations, retrieve for each, merge results

HyDE (Hypothetical Document Embeddings):
  Problem: query-space vectors ≠ document-space vectors
  Fix: generate a hypothetical answer → embed THAT (it's in doc-space)

HYBRID (BM25 + Semantic):
  Problem: semantic search misses exact technical terms ('EN-DE', 'h=8')
  Fix: combine BM25 (keyword matching) + semantic (meaning)

CONTEXTUAL COMPRESSION:
  Problem: 800-char chunk has 600 chars of noise, only 200 are relevant
  Fix: LLM reads chunk, extracts ONLY the relevant sentences
```

In [ ]:
# ── Technique 1: Multi-Query Retriever ──────────────────────────────────────────
#
# PROBLEM: "How many heads in attention?" might miss a chunk that says
#   "The model employs h parallel attention functions" (different vocabulary)
#
# SOLUTION: Generate 3 different phrasings, retrieve for each, deduplicate.
#   Query 1: "How many attention heads?"
#   Query 2: "What is h in multi-head attention?"
#   Query 3: "Parallel attention functions count in transformer"
#   → Union of all retrieved docs → better coverage

from langchain_classic.retrievers.multi_query import MultiQueryRetriever

mq_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    llm=llm,
    include_original=True,    # also include the original query's results
)

# Test
import logging
# Enable to see generated queries
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

test_q = "How many attention heads does the transformer use?"
print(f"Multi-Query Retriever Test")
print(f"Original query: '{test_q}'\n")

mq_docs = mq_retriever.invoke(test_q)

logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.WARNING)

print(f"\nRetrieved {len(mq_docs)} unique chunks (vs {TOP_K} from basic RAG):")
for doc in mq_docs:
    paper = doc.metadata.get("paper_id", "?")
    page  = doc.metadata.get("page", "?")
    print(f"  [{paper} p{page}] {doc.page_content[:90].strip()}...")

In [ ]:
# ── Technique 2: HyDE — Hypothetical Document Embeddings ───────────────────────
#
# INSIGHT: Embedding a question gives a different vector than embedding a passage.
#   Question:  "How many attention heads?"  → [query-space vector]
#   Document:  "We use h=8 parallel heads"  → [document-space vector]
#
# HyDE FIX: Generate a hypothetical answer first, embed THAT.
#   Hypothetical: "The transformer uses 8 attention heads in the base model..."
#   This hypothetical answer is in document-space → better similarity matches!

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

HYDE_PROMPT = ChatPromptTemplate.from_template("""
Write a short hypothetical passage from an academic AI/ML research paper 
that would answer the following question. 
Write as if you are the paper's author. Be technical and specific. 
Keep it under 150 words.

Question: {question}

Hypothetical passage:
""")

def hyde_retrieve(question: str, k: int = TOP_K) -> list:
    """HyDE: Generate hypothetical doc → embed it → retrieve similar real docs."""
    # Step 1: Generate hypothetical answer
    hyp_chain = HYDE_PROMPT | llm | StrOutputParser()
    hypothetical_doc = hyp_chain.invoke({"question": question})

    print(f"  Hypothetical document generated:\n")
    print(f"  {hypothetical_doc[:300]}")
    print()

    # Step 2: Use hypothetical doc as query (it's in document-space)
    docs = vectorstore.similarity_search(hypothetical_doc, k=k)
    return docs, hypothetical_doc


test_q = "What BLEU scores did the Transformer achieve on translation tasks?"
print(f"HyDE Test")
print(f"Query: '{test_q}'\n")
print("-" * 55)

hyde_docs, hyp_doc = hyde_retrieve(test_q)

print(f"Retrieved {len(hyde_docs)} chunks via HyDE:")
for doc in hyde_docs:
    paper = doc.metadata.get("paper_id", "?")
    page  = doc.metadata.get("page", "?")
    ctype = doc.metadata.get("content_type", "?")
    print(f"  [{paper} p{page} | {ctype}] {doc.page_content[:90].strip()}...")

In [ ]:
# ── Technique 3: Hybrid BM25 + Semantic Search ──────────────────────────────────
#
# PROBLEM: Pure semantic search misses exact technical terms.
#   Query: "BLEU score on WMT 2014 EN-DE"
#   Semantic might miss a chunk that has exact string "EN-DE BLEU: 28.4"
#   because the embedding doesn't prioritize exact keyword matching.
#
# SOLUTION: Combine BM25 (keyword) + Semantic (meaning)
#   BM25:     rewards exact keyword matches (like a search engine)
#   Semantic: rewards conceptual similarity
#   Ensemble: weighted combination → best of both worlds

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# BM25 needs the raw documents
bm25_retriever = BM25Retriever.from_documents(all_documents, k=TOP_K)

semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

# Hybrid: 30% BM25 (keyword) + 70% semantic
# Tune these weights based on your domain:
#   - More technical (exact terms matter) → increase BM25 weight
#   - More conceptual queries → increase semantic weight
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.3, 0.7],
)

# Compare basic vs hybrid
test_q = "WMT 2014 English-to-German translation BLEU"
print(f"Hybrid vs Basic Retrieval Comparison")
print(f"Query: '{test_q}'\n")

print("Basic (semantic only):")
basic_docs = vectorstore.similarity_search(test_q, k=3)
for doc in basic_docs:
    print(f"  [{doc.metadata['paper_id']} p{doc.metadata['page']}] {doc.page_content[:90]}...")

print("\nHybrid (BM25 30% + Semantic 70%):")
hybrid_docs = hybrid_retriever.invoke(test_q)
for doc in hybrid_docs[:3]:
    print(f"  [{doc.metadata['paper_id']} p{doc.metadata['page']}] {doc.page_content[:90]}...")

In [ ]:
# ── Technique 4: Contextual Compression ─────────────────────────────────────────
#
# PROBLEM: Retrieved chunks are 800 chars. Only 200 chars are relevant to the query.
#   Noise in context → LLM distracted → worse answers.
#
# SOLUTION: After retrieval, run each chunk through LLM to extract only
#   the sentences relevant to the query. Then pass compressed chunks to LLM.
#
# TRADE-OFF: Better precision, but 1 extra LLM call per retrieved chunk.
#   Worth it for high-stakes questions. Skip for speed-critical applications.

from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers import ContextualCompressionRetriever

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": TOP_K}),
)

test_q = "What is the model size (parameters) of BERT-Large?"
print(f"Contextual Compression Test")
print(f"Query: '{test_q}'\n")

print("Without compression (full 800-char chunks):")
raw_docs = vectorstore.similarity_search(test_q, k=2)
for doc in raw_docs:
    print(f"  [{doc.metadata['paper_id']} p{doc.metadata['page']}] {len(doc.page_content)} chars")
    print(f"  '{doc.page_content[:200].strip()}...'")
    print()

print("With compression (relevant sentences only):")
compressed_docs = compression_retriever.invoke(test_q)
for doc in compressed_docs:
    print(f"  [{doc.metadata['paper_id']} p{doc.metadata['page']}] {len(doc.page_content)} chars")
    print(f"  '{doc.page_content.strip()}'")
    print()

Contextual Compression Test
Query: 'What is the model size (parameters) of BERT-Large?'

Without compression (full 800-char chunks):
  [bert p3] 767 chars
  'the tensor2tensor library.1 Because the use
of Transformers has become common and our im-
plementation is almost identical to the original,
we will omit an exhaustive background descrip-
tion of the m...'

  [bert p8] 785 chars
  'We can see that larger models lead to a strict ac-
curacy improvement across all four datasets, even
for MRPC which only has 3,600 labeled train-
ing examples, and is substantially different from
the...'

With compression (relevant sentences only):
  [bert p3] 163 chars
  'We primarily report results on two model sizes:\nBERTBASE (L=12, H=768, A=12, Total Param-\neters=110M) and BERTLARGE (L=24, H=1024,\nA=16, Total Parameters=340M).'

  [bert p8] 88 chars
  'By contrast, BERTBASE
contains 110M parameters and BERTLARGE con-
tains 340M parameters.'

  [bert p8] 66 chars
  'contains 110M parameters and BER

In [ ]:
# ── Advanced RAG Chain (Best Combination) ────────────────────────────────────────
# PRODUCTION RECOMMENDATION:
#   Use Hybrid retriever (covers keyword + semantic) and RAG prompt.
#   Skip compression for speed; add it for high-precision use cases.

advanced_rag_chain = (
    {"context": hybrid_retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# Side-by-side comparison
test_q = "What datasets and benchmarks were used to evaluate the BERT model?"

print("Side-by-Side: Basic vs Advanced RAG")
print("=" * 70)
print(f"Question: {test_q}\n")

print("── BASIC RAG (semantic only):")
basic_answer = basic_rag_chain.invoke(test_q)
print(basic_answer[:600])

print("\n── ADVANCED RAG (hybrid BM25 + semantic):")
advanced_answer = advanced_rag_chain.invoke(test_q)
print(advanced_answer[:600])

Side-by-Side: Basic vs Advanced RAG
Question: What datasets and benchmarks were used to evaluate the BERT model?

── BASIC RAG (semantic only):


── ADVANCED RAG (hybrid BM25 + semantic):
Based on the provided context, the BERT model was evaluated using several datasets and benchmarks including:

*   **GLUE Benchmark:** The model participated in evaluations on tasks such as MNLI-(m/mm), QQP, QNLI, SST-2, CoLA, STS-B, MRPC, RTE (excluding WNLI) to obtain scores on the official GLUE leaderboard [BERT: Pre-training of Deep Bidirectional Transformers, Page 6].
*   **SQuAD v1.1:** The Stanford Question Answering Dataset was used for question answering tasks involving crowd-sourced question/answer pairs [BERT: Pre-training of Deep Bidirectional Transformers, Page 6].
*   **SWAG:** Ev


---
# Chapter 7 — Agentic RAG with LangGraph (CRAG Pattern)

## The Problem with Chains (Chapters 5-6)

```
Chain (Chapter 5-6):
  retrieve → format → generate
  LINEAR. No checks. No self-correction.

  What if retrieved chunks are totally irrelevant?
  → LLM still generates → may hallucinate

  What if the generated answer is wrong?
  → returned to user with no verification
```

## The CRAG Solution (Corrective RAG)

CRAG adds a self-correction loop:

```
1. Route:  Is retrieval even needed? (simple math → no; paper question → yes)
2. Retrieve: Get top-k chunks via hybrid search
3. Grade:  Are these chunks actually relevant? (binary LLM judge)
           NO → rewrite query → retrieve again (loop)
4. Generate: Produce answer from graded chunks
5. Check:  Is the answer grounded? (no hallucinations?)
           NO → regenerate (with gen_retries limit)
6. Return: Only return when grounded OR retry limit hit
```

## Why LangGraph (not AgentExecutor)?

```
AgentExecutor: fixed linear loop
  → cannot add grade_documents between retrieve and generate
  → cannot add hallucination check after generate
  → cannot set different retry limits for different loops

LangGraph: explicit graph — you control every edge
  → add any node, any conditional edge
  → inspect state at any node
  → pause for human input at any node
```

## The Bug We Fixed (Infinite Loop)

```
ORIGINAL CODE (buggy):
  generate → check_hallucination → 'hallucinated' → generate → ...
  retries was only incremented in transform_query
  but generate↔check loop never called transform_query
  → retries stayed 0 forever → infinite loop (ran for 25+ minutes!)

FIX:
  Added gen_retries counter, incremented inside generate()
  decide_hallucination now checks gen_retries >= MAX_RETRIES → exit
  Also set num_predict=5 on grader LLM → forces yes/no instantly
  Also set GRADE_DOCS=False → skips 5 LLM calls per question
```

In [ ]:
# ── LangGraph State Definition ──────────────────────────────────────────────────
from typing import TypedDict, List, Optional
from langchain_core.documents import Document

class RAGState(TypedDict):
    question      : str              # original user question
    generation    : str              # current LLM answer
    documents     : List[Document]   # retrieved documents
    steps         : List[str]        # trace of nodes visited
    retries       : int              # query-rewrite retry counter
    gen_retries   : int              # generation retry counter  ← NEW (fixes infinite loop)
    route         : str              # "vectorstore" or "direct"


# ── Grader LLM — small context, tiny output, fast ────────────────────────────
# FIX: Use num_predict=5 so the grader LLM emits at most 5 tokens.
# This forces 'yes'/'no' instead of long explanations, making each
# grading call 10-20x faster.
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

grader_llm = ChatOllama(model=LLM_MODEL, temperature=0, num_ctx=1024, num_predict=5)

GRADE_PROMPT = ChatPromptTemplate.from_template("""
Answer ONLY 'yes' or 'no'.

{context}

Answer (yes/no):
""")

grade_chain = GRADE_PROMPT | grader_llm | StrOutputParser()

# ── Router ────────────────────────────────────────────────────────────────────
ROUTER_PROMPT = ChatPromptTemplate.from_template("""
Route this question. Answer ONLY 'vectorstore' or 'direct'.
- 'vectorstore': questions about AI papers (transformers, BERT, RAG, attention)
- 'direct': greetings, simple math, unrelated topics

Question: {question}

Answer (vectorstore/direct):
""")

router_llm    = ChatOllama(model=LLM_MODEL, temperature=0, num_ctx=512, num_predict=5)
router_chain  = ROUTER_PROMPT | router_llm | StrOutputParser()

print("State and chains defined.")
print(f"  grader_llm  : num_predict=5  (forces yes/no — fast)")
print(f"  router_llm  : num_predict=5  (forces single word)")
print(f"  gen_retries : tracks generation loop independently")


State and chains defined.
  RAGState fields: ['question', 'generation', 'documents', 'steps', 'retries', 'route']


In [ ]:
# ── LangGraph Node Functions (Fixed) ────────────────────────────────────────────
#
# KEY FIXES:
#  1. gen_retries incremented every time generate→check_halluc loops
#     → prevents infinite generate↔hallucination loop
#  2. grade_documents is OPTIONAL — skipped if GRADE_DOCS=False
#     → saves 5 LLM calls per question
#  3. All grading uses num_predict=5 LLM → near-instant yes/no

from langchain_core.messages import HumanMessage, ToolMessage

GRADE_DOCS = False   # Set True to enable document relevance grading
                    # (adds ~5 extra LLM calls per question)

def route_query(state: RAGState) -> RAGState:
    """Decide: retrieve from vectorstore or answer directly."""
    question = state["question"]
    route    = router_chain.invoke({"question": question}).strip().lower()
    route    = "direct" if "direct" in route else "vectorstore"
    return {**state,
            "route": route,
            "steps": state.get("steps", []) + [f"route→{route}"]}


def retrieve(state: RAGState) -> RAGState:
    """Retrieve relevant documents using hybrid search."""
    docs = hybrid_retriever.invoke(state["question"])
    return {**state,
            "documents": docs,
            "steps": state["steps"] + [f"retrieve({len(docs)} docs)"]}


def grade_documents(state: RAGState) -> RAGState:
    """
    Filter docs for relevance — each doc costs 1 LLM call.
    SKIPPED when GRADE_DOCS=False (default) for speed.
    """
    if not GRADE_DOCS:
        return {**state,
                "steps": state["steps"] + ["grade(skipped-using-all-docs)"]}

    question = state["question"]
    filtered = []
    for doc in state["documents"]:
        ctx   = f"Question: {question}\nDocument: {doc.page_content[:300]}\nRelevant?"
        score = grade_chain.invoke({"context": ctx}).strip().lower()
        if "yes" in score:
            filtered.append(doc)

    kept = filtered if filtered else state["documents"]   # fallback: keep all if none pass
    return {**state,
            "documents": kept,
            "steps": state["steps"] + [f"grade({len(kept)}/{len(state['documents'])} kept)"]}


def generate(state: RAGState) -> RAGState:
    """Generate answer from retrieved documents."""
    from langchain_core.output_parsers import StrOutputParser
    docs      = state.get("documents", [])
    context   = format_docs(docs) if docs else "No documents retrieved."
    answer    = (RAG_PROMPT | llm | StrOutputParser()).invoke({
        "context": context, "question": state["question"]
    })
    # FIX: increment gen_retries on every generate call so hallucination
    # checker loop has a hard exit after MAX_RETRIES
    return {**state,
            "generation" : answer,
            "gen_retries": state.get("gen_retries", 0) + 1,
            "steps"      : state["steps"] + ["generate"]}


def transform_query(state: RAGState) -> RAGState:
    """Rewrite query to improve retrieval when graded docs were irrelevant."""
    from langchain_core.output_parsers import StrOutputParser
    rewrite_prompt = ChatPromptTemplate.from_template("""
    Rewrite this question using different technical terms for better search retrieval.
    Return ONE line only.
    Original: {question}
    Rewritten:
    """)
    new_q = (rewrite_prompt | llm | StrOutputParser()).invoke(
        {"question": state["question"]}
    ).strip().split("\n")[0]

    return {**state,
            "question": new_q,
            "retries" : state.get("retries", 0) + 1,
            "steps"   : state["steps"] + [f"transform→'{new_q[:45]}...'"]}


def check_hallucination(state: RAGState) -> RAGState:
    """Verify the answer is grounded in retrieved docs."""
    docs       = state.get("documents", [])[:2]   # only check against top-2 docs
    generation = state.get("generation", "")
    ctx = (
        f"Documents:\n{format_docs(docs)}\n\n"
        f"Answer:\n{generation[:400]}\n\n"
        f"Is the answer grounded in the documents? (yes/no)"
    )
    score  = grade_chain.invoke({"context": ctx}).strip().lower()
    result = "grounded" if "yes" in score else "hallucinated"
    return {**state,
            "steps": state["steps"] + [f"hallucination→{result}"]}


# ── Routing functions (conditional edges) ─────────────────────────────────────
def decide_route(state: RAGState) -> str:
    return state.get("route", "vectorstore")


def decide_after_grading(state: RAGState) -> str:
    """After grading: generate if docs exist, else transform query (max retries)."""
    if state.get("documents") or state.get("retries", 0) >= MAX_RETRIES:
        return "generate"
    return "transform_query"


def decide_hallucination(state: RAGState) -> str:
    """
    FIX: Exit if grounded OR if gen_retries >= MAX_RETRIES.
    Before fix: gen_retries was never incremented → infinite loop.
    After fix:  gen_retries++ in generate() → loop exits after MAX_RETRIES.
    """
    steps       = state.get("steps", [])
    gen_retries = state.get("gen_retries", 0)
    last_step   = steps[-1] if steps else ""

    if "grounded" in last_step or gen_retries >= MAX_RETRIES:
        return "END"
    return "generate"   # retry — but gen_retries is now tracking this


print("✅ All node functions defined (infinite-loop bug fixed).")
print(f"   GRADE_DOCS   = {GRADE_DOCS}  (False = skip grading, ~5x faster)")
print(f"   MAX_RETRIES  = {MAX_RETRIES}")
print()
print("   Bug fixed: gen_retries++ in generate() breaks the")
print("   generate→check_hallucination→generate infinite loop.")


✅ All node functions defined.


In [ ]:
# ── Build the LangGraph (CRAG) ──────────────────────────────────────────────────
from langgraph.graph import StateGraph, END

def build_rag_graph():
    workflow = StateGraph(RAGState)

    workflow.add_node("route_query",         route_query)
    workflow.add_node("retrieve",             retrieve)
    workflow.add_node("grade_documents",      grade_documents)
    workflow.add_node("generate",             generate)
    workflow.add_node("transform_query",      transform_query)
    workflow.add_node("check_hallucination",  check_hallucination)

    workflow.set_entry_point("route_query")

    workflow.add_conditional_edges(
        "route_query", decide_route,
        {"vectorstore": "retrieve", "direct": "generate"}
    )
    workflow.add_edge("retrieve", "grade_documents")
    workflow.add_conditional_edges(
        "grade_documents", decide_after_grading,
        {"generate": "generate", "transform_query": "transform_query"}
    )
    workflow.add_edge("transform_query", "retrieve")
    workflow.add_edge("generate", "check_hallucination")
    workflow.add_conditional_edges(
        "check_hallucination", decide_hallucination,
        {"END": END, "generate": "generate"}
    )

    return workflow.compile()


rag_app = build_rag_graph()
print("✅ Agentic RAG graph compiled.")
print()
print("LLM calls per question (approximate):")
print("  route_query       : 1  (tiny: num_predict=5)")
print("  retrieve          : 0  (vector similarity — no LLM)")
print("  grade_documents   : 0  (skipped — GRADE_DOCS=False)")
print("  generate          : 1  (main answer)")
print("  check_hallucination: 1 (tiny: num_predict=5)")
print("  TOTAL             : ~3 LLM calls  (was 10-15+ before)")


✅ LangGraph RAG agent compiled.

Graph nodes:
  • __start__
  • route_query
  • retrieve
  • grade_documents
  • generate
  • transform_query
  • check_hallucination
  • __end__


In [ ]:
# ── Run the Agentic RAG ──────────────────────────────────────────────────────────
import time

def run_rag_agent(question: str, verbose: bool = True) -> dict:
    """Run the self-correcting RAG agent and return final state."""
    initial_state = {
        "question"   : question,
        "generation" : "",
        "documents"  : [],
        "steps"      : [],
        "retries"    : 0,
        "gen_retries": 0,   # ← must be initialised to 0
        "route"      : "",
    }

    t0          = time.time()
    final_state = rag_app.invoke(initial_state)
    elapsed     = time.time() - t0

    if verbose:
        print(f"Q: {question}")
        print(f"\nAgent Steps ({elapsed:.1f}s): {' → '.join(final_state['steps'])}")
        print(f"\nSources used:")
        for doc in final_state.get("documents", []):
            paper = doc.metadata.get("paper_id", "?")
            page  = doc.metadata.get("page", "?")
            ctype = doc.metadata.get("content_type", "?")
            print(f"  [{paper} p{page} | {ctype}]")
        print(f"\nAnswer:\n{final_state['generation']}")

    return final_state


# ── Test questions (start with just 1 for speed check) ───────────────────────
print("Running single question first to verify timing...\n")
print("=" * 70)
result = run_rag_agent("What is the key innovation of the Transformer model over RNNs?")
print()

# Uncomment below for 2nd question once timing looks acceptable:
# print("=" * 70)
# result2 = run_rag_agent("How does BERT differ from traditional left-to-right language models?")


Running single question first to verify timing...



NameError: name 'rag_app' is not defined

---
# Chapter 8 — RAG Evaluation with RAGAS

## Why Evaluation Matters

```
Without evaluation:          With RAGAS:
  'It seems to work well'      'Faithfulness: 0.72 (needs work)'
  → subjective, inconsistent   'Context Precision: 0.89 (good)'
  → can't compare techniques   'Context Recall: 0.61 (retrieval gap)'
  → can't detect regressions   → objective, comparable, actionable
```

## The Four RAGAS Metrics Explained

```
1. FAITHFULNESS (0–1)
   Question: Does the answer contain ONLY facts from the context?
   Low score means: LLM is hallucinating facts not in the retrieved chunks.
   Fix: Add hallucination grader (Chapter 7). Use smaller temperature.

2. ANSWER RELEVANCY (0–1)
   Question: Does the answer actually address the question asked?
   Low score means: Correct facts but off-topic. Prompt engineering needed.
   Fix: Add 'Answer the question directly before elaborating' to prompt.

3. CONTEXT PRECISION (0–1)
   Question: Of the retrieved chunks, how many are actually useful?
   Low score means: Too much noise in retrieval. Chunks retrieved but unused.
   Fix: Reduce TOP_K, use contextual compression, use smaller chunks.

4. CONTEXT RECALL (0–1)
   Question: Did we retrieve ALL the information needed to answer?
   Low score means: Important chunks were missed.
   Fix: Increase TOP_K, use multi-query retrieval, hybrid search.
```

## How RAGAS Uses the LLM as a Judge

```
RAGAS doesn't use fixed rules — it uses an LLM to evaluate.

For Faithfulness:
  Judge prompt: 'Given this context, is this claim supported? yes/no'
  → runs for each sentence in the answer
  → score = (supported sentences) / (total sentences)

We wrap our Ollama LLM for RAGAS:
  ragas_llm = LangchainLLMWrapper(ChatOllama(model=LLM_MODEL))
  → RAGAS will call our local model for all judgments (free!)
```

In [ ]:
# ── RAGAS Setup with Ollama ──────────────────────────────────────────────────────
# RAGAS uses an LLM as a judge. We wrap our Ollama models for RAGAS.

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaEmbeddings
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)

# Wrap Ollama LLM for RAGAS (it uses LangChain under the hood)
ragas_llm = LangchainLLMWrapper(ChatOllama(model=LLM_MODEL, temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model=EMBED_MODEL))

# Initialize metrics with our local models
faithfulness       = Faithfulness(llm=ragas_llm)
answer_relevancy   = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb)
context_precision  = LLMContextPrecisionWithReference(llm=ragas_llm)
context_recall     = LLMContextRecall(llm=ragas_llm)

metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

print("RAGAS Setup")
print(f"  LLM Judge   : {LLM_MODEL} (via Ollama)")
print(f"  Embeddings  : {EMBED_MODEL} (via Ollama)")
print(f"  Metrics     : {len(metrics)}")
for m in metrics:
    print(f"    • {m.__class__.__name__}")
print("\n✅ RAGAS configured for fully local evaluation")

In [ ]:
# ── Build Evaluation Dataset ─────────────────────────────────────────────────────
#
# PRODUCTION TIP: In real projects, ground truth answers come from domain experts.
# Here we write them manually since we know these papers.
# Format: list of (question, ground_truth, [retrieved_contexts], generated_answer)

EVAL_SAMPLES = [
    {
        "question"    : "How many attention heads does the base Transformer model use?",
        "ground_truth": "The base Transformer model uses 8 attention heads (h=8) with a model dimension of 512.",
    },
    {
        "question"    : "What are the two pre-training tasks used in BERT?",
        "ground_truth": "BERT uses two pre-training tasks: Masked Language Model (MLM) where 15% of tokens are masked and predicted, and Next Sentence Prediction (NSP) where the model predicts if sentence B follows sentence A.",
    },
    {
        "question"    : "What is the key advantage of RAG over purely parametric models?",
        "ground_truth": "RAG can access and update non-parametric memory (retrieved documents) without retraining, allowing it to handle knowledge-intensive tasks using up-to-date information from a document store.",
    },
    {
        "question"    : "What optimizer was used to train the original Transformer?",
        "ground_truth": "The original Transformer used the Adam optimizer with custom learning rate scheduling that increases linearly for the first warmup steps then decreases proportionally to the inverse square root of the step number.",
    },
]

# Run RAG to get contexts and answers for each sample
print("Generating RAG responses for evaluation dataset...\n")
eval_data = []

for sample in EVAL_SAMPLES:
    q = sample["question"]
    print(f"  Processing: {q[:60]}...")

    # Get retrieved contexts
    retrieved_docs = hybrid_retriever.invoke(q)
    contexts = [doc.page_content for doc in retrieved_docs]

    # Generate answer
    answer = advanced_rag_chain.invoke(q)

    eval_data.append({
        "user_input"         : q,
        "retrieved_contexts" : contexts,
        "response"           : answer,
        "reference"          : sample["ground_truth"],
    })

print(f"\n✅ Evaluation dataset ready: {len(eval_data)} samples")

### Before running: Expected time and what to expect

```
4 questions × 4 metrics = 16 LLM judge calls minimum.
Each call: 20-60 seconds with qwen3.5 locally.
Total expected time: 5-15 minutes.

This is normal. In production:
  - Run evaluation on a small sample (10-20 questions)
  - Schedule overnight for full evaluation
  - Use faster models for judgment (not your main LLM)

If it's too slow: reduce EVAL_SAMPLES list to 2 questions.
```

In [ ]:
# ── Run RAGAS Evaluation ─────────────────────────────────────────────────────────
from ragas import evaluate
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
import pandas as pd

# Build RAGAS EvaluationDataset
ragas_samples = [
    SingleTurnSample(
        user_input          = d["user_input"],
        retrieved_contexts  = d["retrieved_contexts"],
        response            = d["response"],
        reference           = d["reference"],
    )
    for d in eval_data
]

ragas_dataset = EvaluationDataset(samples=ragas_samples)

print("Running RAGAS evaluation (this calls LLM as judge for each metric)...")
print("Expected time: ~2-5 minutes for 4 samples × 4 metrics\n")

results = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
)

# Display results
df = results.to_pandas()
print("\nRAGAS Evaluation Results:")
print("=" * 70)

# Show per-question scores
score_cols = [c for c in df.columns if c not in ["user_input", "retrieved_contexts", "response", "reference"]]

for i, row in df.iterrows():
    print(f"\nQ{i+1}: {row['user_input'][:60]}...")
    for col in score_cols:
        val = row[col]
        bar = "█" * int(val * 20) + "░" * (20 - int(val * 20))
        status = "✅" if val >= 0.7 else "⚠️ " if val >= 0.5 else "❌"
        print(f"  {status} {col:<30} {val:.3f}  [{bar}]")

# Aggregate means
print("\n" + "=" * 70)
print("AGGREGATE SCORES:")
for col in score_cols:
    mean_val = df[col].mean()
    bar = "█" * int(mean_val * 20) + "░" * (20 - int(mean_val * 20))
    status = "✅" if mean_val >= 0.7 else "⚠️ " if mean_val >= 0.5 else "❌"
    print(f"  {status} {col:<30} {mean_val:.3f}  [{bar}]")

In [ ]:
# ── Visualize RAGAS Results ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

score_cols = [c for c in df.columns if c not in ["user_input", "retrieved_contexts", "response", "reference"]]
questions  = [f"Q{i+1}" for i in range(len(df))]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("RAGAS Evaluation — TechMind Research Assistant", fontsize=14, fontweight="bold")

# ── Plot 1: Heatmap of scores ─────────────────────────────────────────────────
ax1 = axes[0]
score_matrix = df[score_cols].values
im = ax1.imshow(score_matrix, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax1.set_xticks(range(len(score_cols)))
ax1.set_xticklabels([c.replace("_", "\n") for c in score_cols], fontsize=9)
ax1.set_yticks(range(len(questions)))
ax1.set_yticklabels(questions)
ax1.set_title("Score Heatmap (per question)")
plt.colorbar(im, ax=ax1, fraction=0.046)

# Add values
for i in range(len(questions)):
    for j in range(len(score_cols)):
        ax1.text(j, i, f"{score_matrix[i, j]:.2f}",
                 ha="center", va="center", fontsize=10, fontweight="bold",
                 color="black")

# ── Plot 2: Radar/Bar chart of averages ───────────────────────────────────────
ax2 = axes[1]
avg_scores = df[score_cols].mean().values
colors     = ["#2ecc71" if s >= 0.7 else "#f39c12" if s >= 0.5 else "#e74c3c" for s in avg_scores]

bars = ax2.bar(range(len(score_cols)), avg_scores, color=colors, edgecolor="black", linewidth=0.5)
ax2.set_xticks(range(len(score_cols)))
ax2.set_xticklabels([c.replace("_", "\n") for c in score_cols], fontsize=9)
ax2.set_ylim(0, 1.1)
ax2.axhline(y=0.7, color="green", linestyle="--", alpha=0.5, label="Good (0.7)")
ax2.axhline(y=0.5, color="orange", linestyle="--", alpha=0.5, label="Acceptable (0.5)")
ax2.set_ylabel("Score")
ax2.set_title("Average RAGAS Scores")
ax2.legend(loc="lower right")

for bar, score in zip(bars, avg_scores):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f"{score:.2f}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.savefig("ragas_evaluation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Chart saved: ragas_evaluation_results.png")